In [ ]:
!pip install pandas

In [ ]:
import urllib.request
import pandas as pd

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/ops4life/mlops-get-started/main/datasets/HR-Employee-Attrition.csv",
    "HR-Employee-Attrition.csv"
)
# Run prior pipeline steps (ingest + validate) to produce cleaned.csv
df_raw = pd.read_csv("HR-Employee-Attrition.csv")
df_raw.to_csv("cleaned.csv", index=False)
print("Setup complete: cleaned.csv ready.")

In [ ]:
import pandas as pd

def feature_data(df):
    df_fe = df.copy()

    # Encoding
    # ------------------------------------
    # 1. Target column
    df_fe['Attrition'] = df_fe['Attrition'].map({'Yes': 1, 'No': 0}).astype('int')

    # 2. Binary Encoding
    df_fe['OverTime'] = df_fe['OverTime'].map({'Yes': 1, 'No': 0}).astype('Int64')
    df_fe['Gender'] = df_fe['Gender'].map({'Male': 1, 'Female': 0}).astype('Int64')

    # 3. Ordinal encoding
    df_fe['BusinessTravel'] = df_fe['BusinessTravel'].map({'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}).astype('Int64')
    df_fe['MaritalStatus'] = df_fe['MaritalStatus'].map({'Single': 0, 'Married': 1, 'Divorced': 2}).astype('Int64')

    # WorkLifeBalance, JobSatisfaction, EnvironmentSatisfaction, RelationshipSatisfaction,
    # JobLevel, PerformanceRating, JobInvolvement are already numeric (1–N), no encoding needed.

    # Aggregate satisfaction
    df_fe['OverallSatisfaction'] = (
        (df_fe['JobSatisfaction'] + df_fe['EnvironmentSatisfaction'] + df_fe['RelationshipSatisfaction'] + df_fe['WorkLifeBalance']) / 4
    ).round().astype('Int64')
    df_fe = df_fe.drop(columns=['JobSatisfaction', 'EnvironmentSatisfaction', 'RelationshipSatisfaction', 'WorkLifeBalance'])

    # Annual income binning
    df_fe['AnnualIncome'] = pd.cut(
        df_fe['MonthlyIncome'] * 12,
        bins=[0, 25000, 50000, 75000, 100000, float('inf')],
        labels=[0, 1, 2, 3, 4],
        include_lowest=True
    ).astype('Int64')
    df_fe = df_fe.drop(columns=['MonthlyIncome'])

    # Age binning
    df_fe['AgeGroup'] = pd.cut(
        df_fe['Age'],
        bins=[17, 30, 40, 50, 65],
        labels=[1, 2, 3, 4]
    ).astype('Int64')
    df_fe = df_fe.drop(columns=['Age'])

    # Derived features
    # 1. Promotion stagnation: high ratio -> stuck without promotion
    df_fe['PromotionStagnation'] = (df_fe['YearsSinceLastPromotion'] / (df_fe['YearsAtCompany'] + 1)).round(3)

    # 2. Career velocity: job level earned relative to total experience
    df_fe['CareerVelocity'] = (df_fe['JobLevel'] / (df_fe['TotalWorkingYears'] + 1)).round(3)

    # 3. Long commute risk indicator
    df_fe['LongCommute'] = (df_fe['DistanceFromHome'] > 20).astype('Int64')

    # 4. Stable manager relationship
    df_fe['StableManager'] = (df_fe['YearsWithCurrManager'] >= 3).astype('Int64')

    # Drop ID/constant columns, high-cardinality categoricals, and consumed source columns
    drop_cols = [c for c in [
        'EmployeeNumber', 'EmployeeCount', 'StandardHours', 'Over18',
        'JobRole', 'EducationField', 'Department',
        'DistanceFromHome', 'YearsWithCurrManager'
    ] if c in df_fe.columns]
    df_fe = df_fe.drop(columns=drop_cols)

    print(df_fe.tail(10))

    df_fe.to_csv("featured.csv", index=False)

    return df_fe


df = pd.read_csv("cleaned.csv")
feature_data(df)